# Projeto Prático 2 (Parte 1) — Classificação de Conservantes com e sem `Momentum` (`NumPy`)

> **Contexto de Engenharia Química e de Alimentos (Seção 5.9 do Livro-Base):**
> Em uma linha industrial de envase de bebidas, precisamos decidir qual **conservante (`A`, `B` ou `C`)** dosar com base em **4 sensores físico-químicos em linha**:
> - $x_1$: Teor de água | $x_2$: Grau de acidez | $x_3$: Temperatura | $x_4$: Tensão interfacial

### Arquitetura e Codificação *One-Hot* (`4-15-3`)
Usamos 4 entradas, **15 neurônios ocultos** e **3 neurônios de saída** (`A = [1,0,0]`, `B = [0,1,0]`, `C = [0,0,1]`).
O grande foco didático deste projeto é comparar **lado a lado, partindo exatamente dos mesmos pesos iniciais**:
1. **Backpropagation Convencional ($\alpha = 0{,}0$):** Sem inércia mecânica.
2. **Backpropagation com *Momentum* ($\alpha = 0{,}9$):** Acumulando 90% do passo anterior para ganhar velocidade em vales suaves e escapar de mínimos locais!

*(Veja a Parte 2 com `Scikit-Learn` e `Keras` em [`05_projeto2_comparacao_sklearn_keras.ipynb`](05_projeto2_comparacao_sklearn_keras.ipynb)).*

---

## 1. Carregamento Limpo dos 148 Ensaios de Treino e 18 de Teste

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline

# Lê os CSVs limpos diretamente com pd.read_csv()
train_df = pd.read_csv("datasets/projeto-pratico-2.csv")
test_df = pd.read_csv("datasets/conjunto-testes-projeto-pratico-2.csv")

print(f"Amostras de treino: {len(train_df)} | Amostras de teste: {len(test_df)}")
print("\nPrimeiras 5 linhas do Treinamento:")
print(train_df.head())
print("\nPrimeiras 5 linhas do Teste:")
print(test_df.head())

---

## 2. Classe `MLPClassifierManual` com Suporte a `Momentum` ($\alpha$)

Repare na equação de atualização com *Momentum* implementada nas linhas finais do `train`:
$$\Delta \mathbf{W}(t) = \eta \cdot (\boldsymbol{\delta} \, \mathbf{entrada}^T) + \alpha \cdot \Delta \mathbf{W}(t-1)$$
Quando $\alpha = 0{,}0$, recuperamos o Backpropagation convencional puro; quando $\alpha = 0{,}9$, ativamos a inércia mecânica!

In [ ]:
class MLPClassifierManual:
    def __init__(self, input_size, hidden_size, output_size, W1_init=None, W2_init=None, seed=42):
        self.rng = np.random.default_rng(seed)

        # Permite injetar exatamente os mesmos pesos iniciais em dois modelos distintos para comparação justa!
        if W1_init is None:
            self.W1 = self.rng.uniform(-0.5, 0.5, size=(hidden_size, input_size + 1))
        else:
            self.W1 = W1_init.copy()

        if W2_init is None:
            self.W2 = self.rng.uniform(-0.5, 0.5, size=(output_size, hidden_size + 1))
        else:
            self.W2 = W2_init.copy()

    @staticmethod
    def sigmoid(u):
        return 1.0 / (1.0 + np.exp(-u))

    @staticmethod
    def sigmoid_derivative(y):
        return y * (1.0 - y)

    def train(self, X, D, eta=0.1, alpha=0.0, epsilon=1e-6, max_epochs=5000):
        N = X.shape[0]
        mse_history = []

        # Memória da variação anterior de pesos (velocidade/inércia inicial = zero)
        delta_W1_ant = np.zeros_like(self.W1)
        delta_W2_ant = np.zeros_like(self.W2)

        inicio = time.perf_counter()

        for epoch in range(1, max_epochs + 1):
            soma_erro = 0.0
            indices = self.rng.permutation(N)

            for idx in indices:
                x = X[idx].reshape(-1, 1)   # (4x1)
                d = D[idx].reshape(-1, 1)   # (3x1)

                # 1. Propagação Direta (Forward Pass)
                x_b = np.vstack([[-1.0], x])           # (5x1)
                y1 = self.sigmoid(self.W1 @ x_b)       # (15x1)

                y1_b = np.vstack([[-1.0], y1])         # (16x1)
                y2 = self.sigmoid(self.W2 @ y1_b)      # (3x1)

                # Erro quadrático acumulado nas 3 saídas
                e = d - y2
                soma_erro += 0.5 * float(np.sum(e ** 2))

                # 2. Retropropagação (Backward Pass)
                delta2 = e * self.sigmoid_derivative(y2)                      # (3x1)
                delta1 = self.sigmoid_derivative(y1) * (self.W2[:, 1:].T @ delta2)  # (15x1)

                # 3. Atualização com Termo de Inércia (Momentum = alpha)
                dW2 = eta * (delta2 @ y1_b.T) + alpha * delta_W2_ant
                self.W2 = self.W2 + dW2
                delta_W2_ant = dW2

                dW1 = eta * (delta1 @ x_b.T) + alpha * delta_W1_ant
                self.W1 = self.W1 + dW1
                delta_W1_ant = dW1

            mse = soma_erro / N
            mse_history.append(mse)
            if mse <= epsilon:
                tempo_total = time.perf_counter() - inicio
                return mse_history, tempo_total, epoch

        tempo_total = time.perf_counter() - inicio
        return mse_history, tempo_total, max_epochs

    def predict_pos(self, X):
        # Pós-processamento por arredondamento simétrico: >= 0.5 vira 1, < 0.5 vira 0
        respostas = []
        for i in range(X.shape[0]):
            x_b = np.vstack([[-1.0], X[i].reshape(-1, 1)])
            y1 = self.sigmoid(self.W1 @ x_b)
            y1_b = np.vstack([[-1.0], y1])
            y2 = self.sigmoid(self.W2 @ y1_b)
            y_pos = (y2 >= 0.5).astype(int).ravel()
            respostas.append(y_pos)
        return np.array(respostas)

---

## 3. Treinando Dois Modelos Independentes com os Mesmos Pesos de Partida

Criamos `mlp_conv` ($\alpha = 0{,}0$) e `mlp_mom` ($\alpha = 0{,}9$) como **dois objetos separados** para podermos validar e comparar ambos depois sobre as 18 amostras de teste!

In [ ]:
X_train = train_df[["x1", "x2", "x3", "x4"]].to_numpy()
D_train = train_df[["d1", "d2", "d3"]].to_numpy()

# Sorteia um único par de matrizes iniciais para garantir comparação 100% justa
rng_base = np.random.default_rng(42)
W1_inicial = rng_base.uniform(-0.5, 0.5, size=(15, 5))
W2_inicial = rng_base.uniform(-0.5, 0.5, size=(3, 16))

# Modelo 1: Backpropagation Convencional (alpha = 0.0)
mlp_conv = MLPClassifierManual(4, 15, 3, W1_init=W1_inicial, W2_init=W2_inicial, seed=42)
hist_conv, tempo_conv, ep_conv = mlp_conv.train(X_train, D_train, eta=0.1, alpha=0.0, epsilon=1e-6, max_epochs=5000)
print(f"[Convencional alpha=0.0] Épocas: {ep_conv} | MSE final: {hist_conv[-1]:.6f} | Tempo: {tempo_conv:.2f}s")

# Modelo 2: Backpropagation + Momentum (alpha = 0.9) partindo dos MESMOS pesos iniciais!
mlp_mom = MLPClassifierManual(4, 15, 3, W1_init=W1_inicial, W2_init=W2_inicial, seed=42)
hist_mom, tempo_mom, ep_mom = mlp_mom.train(X_train, D_train, eta=0.1, alpha=0.9, epsilon=1e-6, max_epochs=5000)
print(f"[Com Momentum alpha=0.9] Épocas: {ep_mom} | MSE final: {hist_mom[-1]:.6f} | Tempo: {tempo_mom:.2f}s")

---

## 4. Gráficos Não Superpostos de Convergência ($E_M$ por Época)

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 7))

ax1.plot(hist_conv, color="royalblue", linewidth=1.8)
ax1.set_title(f"Backpropagation Convencional (alpha=0.0 | MSE final={hist_conv[-1]:.5f} | {tempo_conv:.2f}s)")
ax1.set_xlabel("Época")
ax1.set_ylabel("MSE")
ax1.grid(True, alpha=0.4)

ax2.plot(hist_mom, color="darkorange", linewidth=1.8)
ax2.set_title(f"Backpropagation + Momentum (alpha=0.9 | MSE final={hist_mom[-1]:.5f} | {tempo_mom:.2f}s)")
ax2.set_xlabel("Época")
ax2.set_ylabel("MSE")
ax2.grid(True, alpha=0.4)

plt.tight_layout()
plt.show()

**Análise Física da Curva:** Repare na velocidade inicial! Enquanto o Backpropagation convencional desce devagar pelos primeiros milhares de épocas, o termo de **inércia (*Momentum* $\alpha = 0{,}9$)** despenca o erro quase verticalmente logo nas primeiras 100 épocas!

---

## 5. Validação Comparativa nas 18 Amostras de Teste (Ambos os Modelos)

In [ ]:
X_test = test_df[["x1", "x2", "x3", "x4"]].to_numpy()
D_test = test_df[["d1", "d2", "d3"]].to_numpy()

def calcular_acuracia_exata(Y_previsto, D_real):
    # Conta quantas linhas bateram exatamente os 3 bits [d1, d2, d3]
    acertos = sum(np.array_equal(Y_previsto[i], D_real[i]) for i in range(len(D_real)))
    return acertos, (acertos / len(D_real)) * 100.0

pred_conv = mlp_conv.predict_pos(X_test)
pred_mom = mlp_mom.predict_pos(X_test)

acertos_conv, acc_conv = calcular_acuracia_exata(pred_conv, D_test)
acertos_mom, acc_mom = calcular_acuracia_exata(pred_mom, D_test)

print(f"Convencional (alpha=0.0): {acertos_conv}/{len(D_test)} acertos ({acc_conv:.2f}%)")
print(f"Com Momentum (alpha=0.9): {acertos_mom}/{len(D_test)} acertos ({acc_mom:.2f}%)\n")

# Exibe tabela comparativa detalhada com o modelo de Momentum
mapa_nomes = {(1, 0, 0): "A", (0, 1, 0): "B", (0, 0, 1): "C"}
relatorio = test_df[["Amostra", "x1", "x2", "x3", "x4"]].copy()
relatorio["Real"] = [mapa_nomes.get(tuple(r), "Indefinido") for r in D_test]
relatorio["Prev_Conv"] = [mapa_nomes.get(tuple(r), "Indefinido") for r in pred_conv]
relatorio["Prev_Mom"] = [mapa_nomes.get(tuple(r), "Indefinido") for r in pred_mom]
print(relatorio.to_string(index=False))

---

## 6. Exercícios Práticos de Fixação

### Exercício 1: Quantas Épocas o *Momentum* Precisa para Atingir `MSE < 0.01`?
Usando as listas `hist_conv` e `hist_mom` já calculadas na memória, descubra em **qual época** cada treinamento cruzou pela primeira vez a marca `MSE < 0.01`!

In [ ]:
# Escreva seu código aqui

<details>
<summary><b>Clique aqui para ver o gabarito comentado</b></summary>

```python
ep_001_conv = next((ep + 1 for ep, val in enumerate(hist_conv) if val < 0.01), None)
ep_001_mom = next((ep + 1 for ep, val in enumerate(hist_mom) if val < 0.01), None)

print(f"Época em que MSE < 0.01 (Convencional): {ep_001_conv}")
print(f"Época em que MSE < 0.01 (Com Momentum): {ep_001_mom}")
```

</details>

### Exercício 2: Decisão por `argmax` vs. Arredondamento Simétrico (`>= 0.5`)
Às vezes, no arredondamento `>= 0.5`, uma amostra ambígua pode gerar `[0, 0, 0]` (nenhuma lâmpada acesa!). Altere a decisão para sempre escolher o neurônio de **maior ativação contínua** (`np.argmax(y2)`) e verifique se todas as 18 amostras recebem um único conservante válido.

In [ ]:
# Escreva seu código aqui

<details>
<summary><b>Clique aqui para ver o gabarito comentado</b></summary>

```python
nomes = ["A", "B", "C"]
escolhas_argmax = []
for i in range(len(X_test)):
    xb = np.vstack([[-1.0], X_test[i].reshape(-1, 1)])
    y1 = mlp_mom.sigmoid(mlp_mom.W1 @ xb)
    y2 = mlp_mom.sigmoid(mlp_mom.W2 @ np.vstack([[-1.0], y1])).ravel()
    escolhas_argmax.append(nomes[int(np.argmax(y2))])

print("Conservantes escolhidos via argmax nas 18 amostras:", escolhas_argmax)
```

</details>